In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

In [2]:
from ydata_profiling import ProfileReport

df = pd.read_csv("../data/KOI_Cumulative_clean.csv")

profile = ProfileReport(
    df,
    title="EDA Report",
    explorative=True
)

 
profile.to_file("eda_report.html")  # Save HTML report

c:\Users\haris\Downloads\Exoplanet-Classification\Exoplanet-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\haris\AppData\Local\Temp\ipykernel_26508\1815861696.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Summarize dataset: 100%|█████████▉| 9751/9754 [3:13:40<00:46, 15.50s/it, Detecting duplicates]                          c:\Users\haris\Downloads\Exoplanet-Classification\Exoplanet-Classification\.venv\Lib\site-packages\ydata_profiling\model\pandas\duplicates_pandas.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

## Load Dataset

Load the original Kepler cumulative dataset that will be cleaned before model training.

In [3]:
df = pd.read_csv("../data/KOI_Cumulative_clean.csv")

print(df.shape)

df.head()

(9564, 140)


,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_fpflag_nt,koi_fpflag_ss,...,koi_dicco_mdec,koi_dicco_mdec_err,koi_dicco_msky,koi_dicco_msky_err,koi_dikco_mra,koi_dikco_mra_err,koi_dikco_mdec,koi_dikco_mdec_err,koi_dikco_msky,koi_dikco_msky_err
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,Done,2018-08-16,CANDIDATE,0,0,...,0.200,0.160,0.200,0.170,0.080,0.130,0.310,0.170,0.320,0.160
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,Done,2018-08-16,CANDIDATE,0,0,...,0.000,0.480,0.390,0.360,0.490,0.340,0.120,0.730,0.500,0.450
2,3,10811496,K00753.01,NaN,CANDIDATE,Done,2018-08-16,CANDIDATE,0,0,...,-0.034,0.070,0.042,0.072,0.002,0.071,-0.027,0.074,0.027,0.074
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,Done,2018-08-16,FALSE POSITIVE,0,1,...,0.147,0.078,0.289,0.079,-0.257,0.072,0.099,0.077,0.276,0.076
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,Done,2018-08-16,CANDIDATE,0,0,...,-0.090,0.180,0.100,0.140,0.070,0.180,0.020,0.160,0.070,0.200


## Dataset Information

Inspect the structure and data types before preprocessing.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Columns: 140 entries, rowid to koi_dikco_msky_err
dtypes: float64(116), int64(7), object(17)
memory usage: 10.2+ MB


## Remove Identifier Columns

Identifier columns uniquely identify observations but do not contribute to prediction.

In [5]:
df = df.dropna(axis=1, how='all')
print(df.shape)

(9564, 121)


In [6]:
identifier_columns = [
    "rowid",
    "kepid",
    "kepoi_name",
    "kepler_name"
]

df.drop(columns=identifier_columns, inplace=True)
print(df.shape)

(9564, 117)


## Missing Value Imputation

- Numerical columns are imputed using the median.
- Categorical columns are imputed using the most frequent value (mode).

In [7]:
numeric_columns = df.select_dtypes(include=np.number).columns

categorical_columns = df.select_dtypes(
    include=['object','string']
).columns

In [8]:
numeric_imputer = SimpleImputer(strategy="median")

df[numeric_columns] = numeric_imputer.fit_transform(
    df[numeric_columns]
)

In [9]:
categorical_imputer = SimpleImputer(strategy="most_frequent")

df[categorical_columns] = categorical_imputer.fit_transform(
    df[categorical_columns]
)

## Encode Target Variable

The target variable is converted into numerical labels for model training.

In [10]:
target_mapping = {
    "CANDIDATE":0,
    "CONFIRMED":1,
    "FALSE POSITIVE":2
}

df["koi_disposition"] = df["koi_disposition"].map(
    target_mapping
)

## Remove Low-Value Categorical Features

Some categorical variables primarily describe the Kepler processing pipeline rather than the physical properties of the observed systems.

In [11]:
low_value_columns = [

    "koi_limbdark_mod",
    "koi_trans_mod",
    "koi_quarters"

]

df.drop(columns=low_value_columns, inplace=True)

print(df.shape)

(9564, 114)


## Remove Metadata and Potential Leakage Features

Several columns contain metadata, free-text information, URLs, or outputs generated by the Kepler processing pipeline rather than intrinsic properties of the observed exoplanet systems. These features do not provide meaningful predictive information and may introduce target leakage.

The following columns were removed:

- `koi_vet_stat` – Vetting status assigned by the pipeline.
- `koi_vet_date` – Date of vetting.
- `koi_pdisposition` – Pipeline-generated planetary disposition, closely related to the target variable.
- `koi_disp_prov` – Source/provider of the disposition.
- `koi_comment` – Free-text comments.
- `koi_tce_delivname` – Pipeline delivery/version identifier.
- `koi_datalink_dvr` – Data validation report link.
- `koi_datalink_dvs` – Data validation summary link.

Removing these columns ensures that the model learns from the underlying astrophysical measurements rather than relying on pipeline-generated metadata or information that could bias the classification.

In [12]:
cols_to_drop = [
    "koi_vet_stat",
    "koi_vet_date",
    "koi_pdisposition",
    "koi_disp_prov",
    "koi_comment",
    "koi_tce_delivname",
    "koi_datalink_dvr",
    "koi_datalink_dvs"
]

df = df.drop(columns=cols_to_drop, errors="ignore")

## One-Hot Encoding

The remaining categorical variables are converted into binary indicator variables.

In [13]:
X = df.drop("koi_disposition", axis=1)
y = df["koi_disposition"]

In [14]:
X = pd.get_dummies(
    X,
    columns=[
        "koi_fittype",
        "koi_parm_prov",
        "koi_sparprov"
    ],
    drop_first=True
)

## Final Dataset Overview

Verify that the dataset contains no missing values and is ready for model training.

In [15]:
print(df.shape)

print()

print(df.isnull().sum().sum())

(9564, 106)

0


In [16]:
print(X.shape)
print(y.shape)

(9564, 110)
(9564,)


In [17]:
df.to_csv("../data/cleaned_koi.csv", index=False)